In [ ]:
# ============================================================
# TOPIK:
# Filtering, Noise Reduction, dan Sharpening
#
# Metode:
# 1. Original
# 2. Mean Filter
# 3. Median Filter
# 4. Gaussian Filter
# 5. Sharpening
#
# OCR menggunakan Tesseract
# ============================================================


# ============================================================
# 1. IMPORT LIBRARY
# ============================================================

import cv2
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pytesseract
from pathlib import Path

try:
    pytesseract.get_tesseract_version()
except pytesseract.TesseractNotFoundError as error:
    raise RuntimeError(
        'Tesseract OCR tidak ditemukan. Install Tesseract di sistem '
        'dan pastikan executable-nya tersedia di PATH.'
    ) from error


# ============================================================
# 2. FOLDER INPUT DAN OUTPUT
# ============================================================

project_dir = Path.cwd()
if not (project_dir / 'foto_citra').is_dir():
    project_dir = project_dir.parent

input_folder = project_dir / 'foto_citra'
output_folder = project_dir / 'tugas5' / 'hasil_filtering_ocr'

if not input_folder.is_dir():
    raise FileNotFoundError(f'Folder input tidak ditemukan: {input_folder}')

output_folder.mkdir(parents=True, exist_ok=True)


# ============================================================
# 4. DAFTAR FILE
# ============================================================

files_list = [
    '01_HighQuality_Enhanced.jpg',
    '02_LowContrast.jpg',
    '03_Blurred.jpg',
    '04_HighNoise.jpg',
    '05_LowResolution_Upsampled.jpg',
    '06_Faded_Underexposed.jpg',
    '07_ColorShift_WarmTint.jpg',
    '08_JPEGCompression_Artifacts.jpg',
    '09_CombinedDegradation.jpg'
]


# ============================================================
# 5. GROUND TRUTH
# ============================================================
# Nomor ijazah yang benar

GROUND_TRUTH = '571012022000056'


# ============================================================
# 6. KOORDINAT CROP
# ============================================================
# Dipertahankan dari kode sebelumnya

x1, x2 = 200, 1000
y1, y2 = 2200, 2400


# ============================================================
# 7. FUNGSI MEMBERSIHKAN HASIL OCR
# ============================================================

def clean_ocr(text):

    # Hanya mengambil angka
    text = re.sub(r'[^0-9]', '', text)

    return text


# ============================================================
# 8. FUNGSI OCR
# ============================================================

def run_ocr(image):

    # Tesseract hanya membaca angka
    config = (
        '--psm 7 '
        '-c tessedit_char_whitelist=0123456789'
    )

    text = pytesseract.image_to_string(
        image,
        config=config
    )

    return clean_ocr(text)


# ============================================================
# 9. HITUNG KARAKTER BENAR
# ============================================================
# Karakter dibandingkan berdasarkan posisi.
#
# Contoh:
# Ground Truth : 57101202000056
# Hasil OCR    : 57101202000058
#
# Semua karakter kecuali karakter terakhir benar.
# ============================================================

def calculate_accuracy(ground_truth, ocr_result):

    correct = 0

    # Bandingkan karakter pada posisi yang sama
    for i in range(min(len(ground_truth), len(ocr_result))):

        if ground_truth[i] == ocr_result[i]:
            correct += 1

    # Akurasi berdasarkan jumlah karakter ground truth
    accuracy = (correct / len(ground_truth)) * 100

    return correct, accuracy


# ============================================================
# 10. DEFINISI FILTER
# ============================================================

def mean_filter(image):

    return cv2.blur(
        image,
        (3, 3)
    )


def median_filter(image):

    return cv2.medianBlur(
        image,
        3
    )


def gaussian_filter(image):

    return cv2.GaussianBlur(
        image,
        (3, 3),
        0
    )


def sharpening(image):

    # Unsharp Masking
    blurred = cv2.GaussianBlur(
        image,
        (0, 0),
        3
    )

    sharpened = cv2.addWeighted(
        image,
        1.5,
        blurred,
        -0.5,
        0
    )

    return sharpened


# ============================================================
# 11. MENAMPUNG HASIL
# ============================================================

results = []


# ============================================================
# 12. PROSES SEMUA GAMBAR
# ============================================================

for filename in files_list:

    print('\n')
    print('=' * 80)
    print('MEMPROSES:', filename)
    print('=' * 80)

    # --------------------------------------------------------
    # Baca gambar
    # --------------------------------------------------------

    path = os.path.join(
        input_folder,
        filename
    )

    img = cv2.imread(str(path))

    if img is None:

        print('Gagal membaca:', filename)
        continue


    # --------------------------------------------------------
    # ROTASI 90 DERAJAT CLOCKWISE
    # --------------------------------------------------------

    img_rotated = cv2.rotate(
        img,
        cv2.ROTATE_90_CLOCKWISE
    )


    # --------------------------------------------------------
    # CROP NOMOR IJAZAH
    # --------------------------------------------------------

    crop = img_rotated[
        y1:y2,
        x1:x2
    ]


    # --------------------------------------------------------
    # GRAYSCALE
    # --------------------------------------------------------

    gray = cv2.cvtColor(
        crop,
        cv2.COLOR_BGR2GRAY
    )


    # ========================================================
    # ENHANCEMENT
    # ========================================================

    images = {

        'Original':
            gray,

        'Mean Filter':
            mean_filter(gray),

        'Median Filter':
            median_filter(gray),

        'Gaussian Filter':
            gaussian_filter(gray),

        'Sharpening':
            sharpening(gray)
    }


    # Nama file tanpa extension
    name = os.path.splitext(filename)[0]


    # ========================================================
    # TAMPILKAN PERBANDINGAN
    # ========================================================

    plt.figure(
        figsize=(20, 5)
    )


    # ========================================================
    # OCR SETIAP METODE
    # ========================================================

    for index, (method, image) in enumerate(images.items()):

        # ----------------------------------------------------
        # OCR
        # ----------------------------------------------------

        ocr_result = run_ocr(image)


        # ----------------------------------------------------
        # Hitung karakter benar dan akurasi
        # ----------------------------------------------------

        correct, accuracy = calculate_accuracy(
            GROUND_TRUTH,
            ocr_result
        )


        # ----------------------------------------------------
        # Simpan hasil ke list
        # ----------------------------------------------------

        results.append({

            'File': filename,

            'Metode': method,

            'Hasil OCR': ocr_result,

            'Karakter Benar': correct,

            'Akurasi (%)': round(
                accuracy,
                2
            )
        })


        # ----------------------------------------------------
        # Tampilkan gambar
        # ----------------------------------------------------

        plt.subplot(
            1,
            5,
            index + 1
        )

        plt.imshow(
            image,
            cmap='gray'
        )

        plt.title(
            f'{method}\n'
            f'OCR: {ocr_result}\n'
            f'Akurasi: {accuracy:.2f}%'
        )

        plt.axis('off')


        # ----------------------------------------------------
        # Simpan hasil image
        # ----------------------------------------------------

        method_filename = (
            method
            .lower()
            .replace(' ', '_')
        )

        output_path = os.path.join(
            output_folder,
            f'{name}_{method_filename}.jpg'
        )

        cv2.imwrite(
            output_path,
            image
        )


    # ========================================================
    # SIMPAN GAMBAR PERBANDINGAN
    # ========================================================

    plt.suptitle(
        f'{name} - Perbandingan Filtering dan Sharpening',
        fontsize=14
    )

    plt.tight_layout()

    comparison_path = os.path.join(
        output_folder,
        f'{name}_perbandingan.jpg'
    )

    plt.savefig(
        comparison_path,
        dpi=150,
        bbox_inches='tight'
    )

    plt.show()


# ============================================================
# 13. BUAT DATAFRAME HASIL OCR
# ============================================================

df = pd.DataFrame(results)


# ============================================================
# 14. TAMPILKAN HASIL LENGKAP
# ============================================================

print('\n')
print('=' * 100)
print('HASIL OCR SELURUH CITRA')
print('=' * 100)

display(df)


# ============================================================
# 15. TABEL SESUAI FORMAT TUGAS
# ============================================================


print("\n")
print("=" * 100)
print("TABEL HASIL OCR PER GAMBAR")
print("=" * 100)

# Iterasi setiap file untuk menampilkan tabel terpisah
for filename, group in df.groupby("File", sort=False):
    print(f"\n📄 File: {filename}")

    # Mengambil kolom yang relevan tanpa mengulang nama file di dalam tabel
    tabel_per_file = group[
        ["Metode", "Hasil OCR", "Karakter Benar", "Akurasi (%)"]
    ].reset_index(drop=True)

    display(tabel_per_file)
    print("-" * 60)


# ============================================================
# 16. RATA-RATA AKURASI SETIAP METODE
# ============================================================

summary = (
    df
    .groupby('Metode')
    .agg(
        Rata_rata_Akurasi=('Akurasi (%)', 'mean'),
        Total_Karakter_Benar=('Karakter Benar', 'sum')
    )
    .reset_index()
)

summary[
    'Rata_rata_Akurasi'
] = summary[
    'Rata_rata_Akurasi'
].round(2)


print('\n')
print('=' * 100)
print('RINGKASAN AKURASI SETIAP METODE')
print('=' * 100)

display(summary)


# ============================================================
# 17. GRAFIK PERBANDINGAN AKURASI
# ============================================================

plt.figure(
    figsize=(10, 5)
)

plt.bar(
    summary['Metode'],
    summary['Rata_rata_Akurasi']
)

plt.xlabel('Metode')
plt.ylabel('Rata-rata Akurasi OCR (%)')

plt.title(
    'Perbandingan Rata-rata Akurasi OCR'
)

plt.ylim(
    0,
    100
)

plt.xticks(
    rotation=30
)

plt.tight_layout()

plt.savefig(
    os.path.join(
        output_folder,
        'perbandingan_akurasi.jpg'
    ),
    dpi=150
)

plt.show()


# ============================================================
# 18. SIMPAN TABEL KE CSV
# ============================================================

csv_path = os.path.join(
    output_folder,
    'hasil_ocr.csv'
)

df.to_csv(
    csv_path,
    index=False
)


# ============================================================
# 19. SELESAI
# ============================================================

print('\n')
print('=' * 80)
print('SELESAI')
print('=' * 80)

print(
    'Semua hasil tersimpan di:'
)

print(
    output_folder
)

print(
    '\nGround Truth:',
    GROUND_TRUTH
)